# 00 · Setup check

Run this once before the talk: the keys exist (their values are never printed), the packages are
the tested versions, Ollama answers, the demo database is built, and the MCP server starts.

In [ ]:
# Reload the src modules automatically when they change (no kernel restart needed).
%load_ext autoreload
%autoreload 2

In [ ]:
import os

from dotenv import load_dotenv

load_dotenv()
for name in ["OPENAI_API_KEY", "ANTHROPIC_API_KEY", "OLLAMA_BASE_URL"]:
    print(f"{name:18} {'set' if os.environ.get(name) else 'MISSING'}")

`.env` is loaded into the environment; the notebooks only ever check that a key is set.

In [ ]:
from importlib.metadata import version

packages = [
    "langchain",
    "langgraph",
    "pydantic-ai-slim",
    "strands-agents",
    "mcp",
    "openai",
    "anthropic",
]
for package in packages:
    print(f"{package:18} {version(package)}")

The brief lists the tested versions; a newer release is fine until something breaks.

In [ ]:
import httpx

ollama_url = os.environ["OLLAMA_BASE_URL"].removesuffix("/v1")
models = httpx.get(f"{ollama_url}/api/tags", timeout=5).json()["models"]
print([model["name"] for model in models])

Stage 3 switches to `qwen3:8b`, so it must be in this list (`ollama pull qwen3:8b`).

In [ ]:
import subprocess
import sys

from agentic_demo.paths import ROOT

_ = subprocess.run([sys.executable, ROOT / "data" / "seed.py"], check=True)

The seed script writes 216 synthetic rows from a fixed seed and fills the expected answers in `data/questions.yaml`.

In [ ]:
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

from agentic_demo.paths import SERVER_PATH

server = StdioServerParameters(command=sys.executable, args=[str(SERVER_PATH)])
async with stdio_client(server) as (read, write), ClientSession(read, write) as session:
    await session.initialize()
    print([tool.name for tool in (await session.list_tools()).tools])
    result = await session.call_tool(
        "run_sql", {"query": "SELECT COUNT(*) AS rows FROM sales"}
    )
    print(result.structured_content)

The same server, started the same way, serves all three frameworks.